# GeoSure API fetch and load

In [44]:
import requests
import json
import pandas as pd
import time
from datetime import date
import configparser
import pyexasol

In [45]:
def get_token(urlLogin):
    """Function to retrieve token and Refresh token"""
    auth = {"client_id": "00000000-0000-0000-0000-000000000000", "client_secret": "00000000-0000-0000-0000-000000000000"}
    response = requests.post(urlLogin, json=auth)
    if response.status_code == 200:
        data = response.text
        parsed = json.loads(data)
        token = parsed['token']
        refreshToken = parsed['refreshToken']
    
    return token, refreshToken

In [ ]:
def get_refresh(url, refreshToken):
    """Function to retrieve token given a Refresh token"""
    urlRefresh = url+'/auth/refresh'
    refresh = {"client_id": "00000000-0000-0000-0000-000000000000", 
               "client_secret": "00000000-0000-0000-0000-000000000000",
               "refreshToken": refreshToken}
    
    response = requests.post(urlRefresh, json=refresh)
    if response.status_code == 200:
        data = response.text
        parsed = json.loads(data)
        newToken = parsed['token']
    else:
        print('Failed to retrieve token')
    return newToken

In [3]:
def GeoSureAPIFetch(urlAuth):
    """Main function to fetch GeoSure hotels"""
    token, refreshToken = get_token(urlAuth)
    startTime = time.time()
    urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page=1&size=500"
    getResponse = requests.get(urlGet)
    rename_columns ={'hkey':'HOTEL_ID', 'gs_id':'GS_ID','gs_distance':'GS_DISTANCE',
                     'date': 'GS_DATE','gs_province': 'GS_PROVINCE_NAME','gs_region':'GS_REGION_NAME',
                      'gs_district': 'GS_DISTRICT_NAME','gs_city':'GS_CITY_NAME','gs_country':'GS_COUNTRY_NAME',
                      'gs_type':'GS_TYPE','gs_countrycode':'GS_COUNTRY_ISO_A2','gs_population':'GS_POPULATION',
                      'composite':'GS_COMPOSITE_SCORE','nightime':'NIGHT_SAFE_SCORE', 'physical':'PHYSICAL_SAFE_SCORE', 
                      'women':'WOMEN_SAFE_SCORE', 'theft':'THEFT_SCORE','freedom':'BASIC_FREEDOM_SCORE', 'health':'HEALTH_MEDS_SCORE',
                      'lgbtq':'LGBTQ_SAFE_SCORE', 'type':'TYPE', 'status':'GS_STATUS'}
    ### Initialize the dataframe
    df = pd.DataFrame()
    if getResponse.status_code == 200:
        data = getResponse.text
        parsed = json.loads(data)
        print(f"Status code: {getResponse.status_code}")
    else: 
        print("Error in fetching the pages")

    try:
        for x in range(int(parsed['total_pages'])):
            
            if x in (300, 600, 900, 1200):
                token = get_refresh(refreshToken)
                print('Retrieving new token')

            urlIter = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page={x+1}&size=500"
            getResponse = requests.get(urlIter)

            if getResponse.status_code == 200:
                data = getResponse.text
                parsed = json.loads(data)
                dfTemp = pd.DataFrame(parsed.get('results'))
                df = df.append(dfTemp, ignore_index=True)
            else: 
                print("Request failed page: {} ".format(x))
        df['date']= df['date'].astype(str).str[:-6]
        df['date']= pd.to_datetime(df['date'], utc=False)
        df['date']= pd.to_datetime(df['date'], format='%Y%m%d', errors='ignore').dt.date
        df['LDTS']= time.strftime('%Y-%m-%d')
        df = df[df.hkey.notnull()]
        df.rename(columns = rename_columns, inplace = True)
        print(f'Number of records having HOTEL_IDs: {df.HOTEL_ID.nunique()}')
        endTime = time.time() - startTime
        print(f'Time in minutes: {round(endTime/60,2)}')
    except Exception as e:
        endTime = time.time() - startTime
        print('Failed in function GeosureFetch - ')
        print(f'Time in minutes: {round(endTime/60,2)}')
        raise e
    return df

In [4]:
def GeoSureLoadDet(df): 
    """Function to load GeoSure hotels in DET environment"""
    harmonized_df = df
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\USER\\.spyder-py3\\pfxDET.ini')
        dsn=config['pfxDET']['dsn']
        user=config['pfxDET']['user']
        pwd=config['pfxDET']['pwd']
        schema=config['pfxDET']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.GEOSURE_HOTELS")
        connect.import_from_pandas(harmonized_df, table = ('DWHPFX','GEOSURE_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except Exception as e:
        print('Failed in function GeoSureLoad - ')
        raise e

In [5]:
def GeoSureLoadProd(df):
"""Function to load GeoSure hotels in PROD environment"""
    harmonized_df = pd.DataFrame()
    harmonized_df = df
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\USER\\.spyder-py3\\pfxPROD.ini')
        dsn=config['pfxPROD']['dsn']
        user=config['pfxPROD']['user']
        pwd=config['pfxPROD']['pwd']
        schema=config['pfxPROD']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.GEOSURE_HOTELS")
        connect.import_from_pandas(harmonized_df, table = ('DWHPFX','GEOSURE_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except:
        print('Failed in function cleanNsafeLoad - check DB connection')
        raise e

In [6]:
df = GeoSureAPIFetch('https://api.hotel-audit.hrs.com/auth/login')

In [6]:
GeoSureLoadDet(df)
GeoSureLoadProd(df)

In [ ]:
# pd.options.display.max_columns = None
# df.reset_index(drop=True, inplace=True)
# df.head()

# Testing Ground below 

In [12]:
urlAuth= 'https://api.hotel-audit.hrs.com/auth/login'
token, refreshToken = get_token(urlAuth)
startTime = time.time()
urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page=1&size=500"
getResponse = requests.get(urlGet)
rename_columns ={'hkey':'HOTEL_ID', 'gs_id':'GS_ID','gs_distance':'GS_DISTANCE',
                 'date': 'GS_DATE','gs_province': 'GS_PROVINCE_NAME','gs_region':'GS_REGION_NAME',
                  'gs_district': 'GS_DISTRICT_NAME','gs_city':'GS_CITY_NAME','gs_country':'GS_COUNTRY_NAME',
                  'gs_type':'GS_TYPE','gs_countrycode':'GS_COUNTRY_ISO_A2','gs_population':'GS_POPULATION',
                  'composite':'GS_COMPOSITE_SCORE','nightime':'NIGHT_SAFE_SCORE', 'physical':'PHYSICAL_SAFE_SCORE', 
                  'women':'WOMEN_SAFE_SCORE', 'theft':'THEFT_SCORE','freedom':'BASIC_FREEDOM_SCORE', 'health':'HEALTH_MEDS_SCORE',
                  'lgbtq':'LGBTQ_SAFE_SCORE', 'type':'TYPE', 'status':'GS_STATUS'}
### Initialize the dataframe
df = pd.DataFrame()
if getResponse.status_code == 200:
    data = getResponse.text
    parsed = json.loads(data)
    print(f"Status code: {getResponse.status_code}")
else: 
    print("Error in fetching the pages")

In [60]:
token, refreshToken = get_token(urlAuth)
startTime = time.time()
urlGet = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page=1&size=500"
getResponse = requests.get(urlGet)
rename_columns ={'hkey':'HOTEL_ID', 'gs_id':'GS_ID','gs_distance':'GS_DISTANCE',
                 'date': 'GS_DATE','gs_province': 'GS_PROVINCE_NAME','gs_region':'GS_REGION_NAME',
                  'gs_district': 'GS_DISTRICT_NAME','gs_city':'GS_CITY_NAME','gs_country':'GS_COUNTRY_NAME',
                  'gs_type':'GS_TYPE','gs_countrycode':'GS_COUNTRY_ISO_A2','gs_population':'GS_POPULATION',
                  'composite':'GS_COMPOSITE_SCORE','nightime':'NIGHT_SAFE_SCORE', 'physical':'PHYSICAL_SAFE_SCORE', 
                  'women':'WOMEN_SAFE_SCORE', 'theft':'THEFT_SCORE','freedom':'BASIC_FREEDOM_SCORE', 'health':'HEALTH_MEDS_SCORE',
                  'lgbtq':'LGBTQ_SAFE_SCORE', 'type':'TYPE', 'status':'GS_STATUS'}
### Initialize the dataframe
df = pd.DataFrame()
if getResponse.status_code == 200:
    data = getResponse.text
    parsed = json.loads(data)
    print(f"Status code: {getResponse.status_code}")
else: 
    print("Error in fetching the pages")

try:
    loopstart = time.time()
    for x in range(int(parsed['total_pages'])):
        
        if (loopstart -time.time())>60:
            token, refreshToken = get_token(urlAuth)
            loopstart = time.time()
            print('Retrieving new token')

        urlIter = f"https://api.hotel-audit.hrs.com/v2/audits/report/geosure?token={token}&page={x+1}&size=500"
        getResponse = requests.get(urlIter)
        
        if getResponse.status_code == 200:
            data = getResponse.text
            parsed = json.loads(data)
            dfTemp = pd.DataFrame(parsed.get('results'))
            df = df.append(dfTemp, ignore_index=True)
            print("Page: {} ".format(x))
        else: 
            print("Request failed page: {} ".format(x))
    df['date']= df['date'].astype(str).str[:-6]
    df['date']= pd.to_datetime(df['date'], utc=False)
    df['date']= pd.to_datetime(df['date'], format='%Y%m%d', errors='ignore').dt.date
    df['LDTS']= time.strftime('%Y-%m-%d')
    df = df[df.hkey.notnull()]
    df.rename(columns = rename_columns, inplace = True)
    print(f'Number of records having HOTEL_IDs: {df.HOTEL_ID.nunique()}')
    endTime = time.time() - startTime
    print(f'Time in minutes: {round(endTime/60,2)}')
except Exception as e:
    endTime = time.time() - startTime
    print('Failed in function GeosureFetch - ')
    print(f'Time in minutes: {round(endTime/60,2)}')
    raise e

In [48]:
df.shape

In [55]:
url = 'https://api.hotel-audit.hrs.com'
token, refreshToken = get_token(urlAuth)
urlRefresh = url+'/auth/refresh'

refresh = {"client_id": "00000000-0000-0000-0000-000000000000", 
           "client_secret": "00000000-0000-0000-0000-000000000000",
           "refreshToken": refreshToken}
response = requests.post(urlRefresh, json=refresh)
if response.status_code == 200:
    data = response.text
    parsed = json.loads(data)
    token = parsed['token']
    print(token)
else:
    print('fail')

In [57]:
import time
starttime = time.time()
while (time.time() - starttime)>10:
    print("tick")
    time.sleep(10.0 - ((time.time() - starttime) % 10.0))

In [59]:
(100-80)%10